In [2]:
!pip install crewai==0.28.8 \
            crewai_tools==0.1.6 \
            langchain==0.1.14 \
            langchain_community==0.0.30 \
            openai==1.25.1 \
            httpx==0.27.0 \
            python-dotenv

In [4]:
import pandas as pd
import numpy as np

PATHS = {
    "production_plan": "./A_company_crewai_demo_csvs/production_plan.csv",
    "process_master": "./A_company_crewai_demo_csvs/process_master.csv",
    "line_assignment": "./A_company_crewai_demo_csvs/line_assignment.csv",
    "work_event_log": "./A_company_crewai_demo_csvs/work_event_log.csv",
    "historical_ct": "./A_company_crewai_demo_csvs/historical_ct.csv",
    "delay_history": "./A_company_crewai_demo_csvs/delay_history.csv",
    "real_time_production": "./A_company_crewai_demo_csvs/real_time_production.csv",
}

dfs = {k: pd.read_csv(v) for k, v in PATHS.items()}

plan = dfs["production_plan"].copy()
pm   = dfs["process_master"].copy()
la   = dfs["line_assignment"].copy()
wel  = dfs["work_event_log"].copy()
hct  = dfs["historical_ct"].copy()
dh   = dfs["delay_history"].copy()
rt   = dfs["real_time_production"].copy()

# 타입 정리
plan["due_time"] = pd.to_datetime(plan["due_time"])
wel["event_time"] = pd.to_datetime(wel["event_time"])
la["planned_start"] = pd.to_datetime(la["planned_start"])
la["planned_end"]   = pd.to_datetime(la["planned_end"])
rt["time"] = pd.to_datetime(rt["time"])

plan


,date,shift,po_id,sku,sku_name,target_qty,due_time,priority,customer_grade,notes
0,2026-01-15,DAY,PO20260115-004,SKU-SRM-30,세럼 30ml,300,2026-01-15 14:00:00,1,B,NaN
1,2026-01-15,DAY,PO20260115-002,SKU-TNR-200,토너 200ml,360,2026-01-15 15:00:00,1,A,NaN
2,2026-01-15,DAY,PO20260115-001,SKU-CLN-100,클렌저 100ml,300,2026-01-15 14:00:00,3,A,NaN
3,2026-01-15,DAY,PO20260115-003,SKU-CRM-50,크림 50ml,540,2026-01-15 14:00:00,3,C,NaN
4,2026-01-15,DAY,PO20260115-005,SKU-SPF-50,선크림 50ml,360,2026-01-15 14:00:00,3,B,NaN


In [18]:
from dataclasses import dataclass
from typing import Dict, Any, List, Optional

STEP_ORDER = (
    pm[["sku","step_id","step_order","step_name","std_ct_sec"]]
    .drop_duplicates()
)

def get_snapshot(snapshot_time: Optional[pd.Timestamp] = None) -> pd.DataFrame:
    """15분 단위 스냅샷에서 특정 시점(없으면 최신) 추출"""
    if snapshot_time is None:
        snapshot_time = rt["time"].max()
    snap = rt[rt["time"] == snapshot_time].copy()
    return snap, snapshot_time

def get_last_step_state(po_id: str, asof: pd.Timestamp) -> Dict[str, Any]:
    """event log 기반: 특정 PO가 asof 시점에 어떤 step에서 어떤 상태인지 추정"""
    sub = wel[(wel["po_id"] == po_id) & (wel["event_time"] <= asof)].sort_values("event_time")
    if sub.empty:
        return {"po_id": po_id, "state": "NO_LOG"}

    last_by_step = sub.groupby("step_id").tail(1)

    # 진행 중(step ongoing) 판단: 마지막 이벤트가 start/resume/pause 이고 이후 end가 없으면 ongoing
    ongoing = []
    for _, r in last_by_step.iterrows():
        if r["event_type"] in ("start","resume","pause"):
            ended = sub[(sub["step_id"] == r["step_id"]) & (sub["event_type"] == "end")]
            if ended.empty or ended["event_time"].max() < r["event_time"]:
                ongoing.append(r)

    sku = sub.iloc[-1]["sku"]
    om = STEP_ORDER[STEP_ORDER["sku"] == sku].set_index("step_id")["step_order"].to_dict()

    if ongoing:
        ongoing.sort(key=lambda r: om.get(r["step_id"], 0), reverse=True)
        r = ongoing[0]
        return {
            "po_id": po_id,
            "sku": r["sku"],
            "line_id": r["line_id"],
            "step_id": r["step_id"],
            "event_type": r["event_type"],
            "event_time": r["event_time"],
            "operator_id": r["operator_id"],
            "memo": r.get("memo", None),
        }

    # ongoing 없으면 마지막 end 기준으로 "최근 완료 step"
    r = sub[sub["event_type"] == "end"].tail(1)
    if r.empty:
        r = sub.tail(1)
    r = r.iloc[0]
    return {
        "po_id": po_id,
        "sku": r["sku"],
        "line_id": r["line_id"],
        "step_id": r["step_id"],
        "event_type": "end",
        "event_time": r["event_time"],
        "operator_id": r["operator_id"],
        "memo": r.get("memo", None),
    }

def detect_delayed_start(asof: pd.Timestamp, threshold_min: int = 10) -> pd.DataFrame:
    """
    계획(planned_start) 대비 실제 start가 늦은 step 감지
    """
    # 실제 start 시간
    starts = wel[wel["event_type"] == "start"][["po_id","step_id","event_time"]].rename(columns={"event_time":"actual_start"})
    merged = la.merge(starts, on=["po_id","step_id"], how="left")

    merged = merged[merged["planned_start"] <= asof].copy()
    merged["delay_min"] = (merged["actual_start"] - merged["planned_start"]).dt.total_seconds() / 60
    merged["delay_min"] = merged["delay_min"].fillna(np.nan)

    out = merged[(merged["actual_start"].isna()) | (merged["delay_min"] >= threshold_min)].copy()
    out = out.sort_values(["po_id","planned_start"])
    return out[["po_id","sku","line_id","step_id","operator_id","planned_start","actual_start","delay_min"]]

def detect_pause(asof: pd.Timestamp) -> pd.DataFrame:
    """pause 상태 PO/step 감지"""
    rows = []
    for po in plan["po_id"].unique():
        st = get_last_step_state(po, asof)
        if st.get("event_type") == "pause":
            rows.append(st)
    return pd.DataFrame(rows)

def build_risk_ranking(asof: pd.Timestamp, top_n: int = 5) -> pd.DataFrame:
    """
    간단 PoC용 리스크 스코어:
    - progress 낮을수록 +
    - due_time 임박/초과일수록 +
    - pause면 크게 +
    - priority(1이 제일 중요) 반영
    """
    snap, _ = get_snapshot(asof)
    snap_map = snap.set_index("po_id")[["progress_pct","cumulative_qty","target_qty","status"]]

    rows = []
    for _, r in plan.iterrows():
        po = r["po_id"]
        due = r["due_time"]
        min_to_due = (due - asof).total_seconds()/60

        if po in snap_map.index:
            prog = float(snap_map.loc[po, "progress_pct"])
        else:
            prog = np.nan

        st = get_last_step_state(po, asof)
        paused = (st.get("event_type") == "pause")

        base = (100 - prog) if not np.isnan(prog) else 30
        due_pen = max(0, -min_to_due) + max(0, 30 - min_to_due)  # 이미 늦음 + 30분 이내 임박
        pause_pen = 50 if paused else 0
        pri_pen = (4 - int(r["priority"])) * 5  # priority=1이면 15, priority=3이면 5

        score = base + due_pen + pause_pen + pri_pen

        rows.append({
            "po_id": po,
            "sku": r["sku"],
            "due_time": due,
            "min_to_due": round(min_to_due, 1),
            "progress_pct": None if np.isnan(prog) else round(prog, 1),
            "current_step": st.get("step_id"),
            "current_event": st.get("event_type"),
            "line_id": st.get("line_id"),
            "operator_id": st.get("operator_id"),
            "memo": st.get("memo"),
            "risk_score": round(score, 1),
        })

    out = pd.DataFrame(rows).sort_values("risk_score", ascending=False).head(top_n)
    return out

def summarize_field_status(asof: pd.Timestamp) -> str:
    """관리자가 보는 현장요약 텍스트(웹 UI 카드에 바로 띄울 정도)"""
    snap, asof = get_snapshot(asof)
    lines = []
    lines.append(f"[현장 요약] 기준시각: {asof:%Y-%m-%d %H:%M}")
    for _, r in snap.sort_values("progress_pct").iterrows():
        po = r["po_id"]
        st = get_last_step_state(po, asof)
        step_name = STEP_ORDER[(STEP_ORDER["sku"]==r["sku"]) & (STEP_ORDER["step_id"]==st.get("step_id"))]["step_name"]
        step_name = step_name.iloc[0] if len(step_name) else st.get("step_id")
        memo = st.get("memo")
        memo_txt = f" / 메모: {memo}" if isinstance(memo, str) and memo.strip() else ""
        lines.append(
            f"- {po}({r['sku_name']}): {r['progress_pct']:.1f}% / 현재: {step_name}({st.get('event_type')})"
            f" / 라인:{st.get('line_id')} / 작업자:{st.get('operator_id')}{memo_txt}"
        )
    return "\n".join(lines)


In [6]:
ASOF = pd.Timestamp("2026-01-15 14:00:00")

print(summarize_field_status(ASOF))

print("\n[지연 시작 감지 - planned_start 대비]")
display(detect_delayed_start(ASOF, threshold_min=10))

print("\n[PAUSE 감지]")
display(detect_pause(ASOF))

print("\n[리스크 TOP 랭킹]")
display(build_risk_ranking(ASOF, top_n=5))


[현장 요약] 기준시각: 2026-01-15 14:00
- PO20260115-003(크림 50ml): 77.8% / 현재: 충진(Filling)(pause) / 라인:L2-충진 / 작업자:OP08 / 메모: 원료 용기 부족(자재 지연)
- PO20260115-001(클렌저 100ml): 100.0% / 현재: 라벨링(start) / 라인:L3-포장 / 작업자:OP04
- PO20260115-002(토너 200ml): 100.0% / 현재: 라벨링(start) / 라인:L3-포장 / 작업자:OP04
- PO20260115-004(세럼 30ml): 100.0% / 현재: 캡핑/실링(start) / 라인:L2-충진 / 작업자:OP03
- PO20260115-005(선크림 50ml): 100.0% / 현재: 캡핑/실링(start) / 라인:L2-충진 / 작업자:OP08

[지연 시작 감지 - planned_start 대비]


,po_id,sku,line_id,step_id,operator_id,planned_start,actual_start,delay_min



[PAUSE 감지]


,po_id,sku,line_id,step_id,event_type,event_time,operator_id,memo
0,PO20260115-003,SKU-CRM-50,L2-충진,S30,pause,2026-01-15 13:53:00,OP08,원료 용기 부족(자재 지연)



[리스크 TOP 랭킹]


,po_id,sku,due_time,min_to_due,progress_pct,current_step,current_event,line_id,operator_id,memo,risk_score
3,PO20260115-003,SKU-CRM-50,2026-01-15 14:00:00,0.0,77.8,S30,pause,L2-충진,OP08,원료 용기 부족(자재 지연),107.2
0,PO20260115-004,SKU-SRM-30,2026-01-15 14:00:00,0.0,100.0,S40,start,L2-충진,OP03,NaN,45.0
2,PO20260115-001,SKU-CLN-100,2026-01-15 14:00:00,0.0,100.0,S50,start,L3-포장,OP04,NaN,35.0
4,PO20260115-005,SKU-SPF-50,2026-01-15 14:00:00,0.0,100.0,S40,start,L2-충진,OP08,NaN,35.0
1,PO20260115-002,SKU-TNR-200,2026-01-15 15:00:00,60.0,100.0,S50,start,L3-포장,OP04,NaN,15.0


In [ ]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

api_key = os.getenv("OPENAI_API_KEY")  # .env 파일에서 로드 # 개인 API 키를 입력합니다.

# 키가 없을 경우 예외 발생
if not api_key:
    raise ValueError("API 키가 비어 있습니다. 올바른 키를 입력하세요.")

os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o'

In [19]:
from crewai_tools import tool

# 전역 변수로 asof를 주입할 거라, 일단 None으로 선언
ASOF_FOR_TOOLS = None

@tool("field_status")
def field_status_tool() -> str:
    """현장(라인/작업자/단계/진척)을 요약해 반환"""
    return summarize_field_status(ASOF_FOR_TOOLS)

@tool("delayed_start")
def delayed_start_tool() -> str:
    """계획 대비 지연 시작/미시작 공정 목록을 반환"""
    df = detect_delayed_start(ASOF_FOR_TOOLS, threshold_min=10)
    return df.to_string(index=False) if len(df) else "지연 시작/미시작 없음"

@tool("pause_status")
def pause_tool() -> str:
    """pause 상태 공정 목록을 반환"""
    df = detect_pause(ASOF_FOR_TOOLS)
    return df.to_string(index=False) if len(df) else "pause 없음"

@tool("risk_rank")
def risk_rank_tool() -> str:
    """리스크 TOP 랭킹을 반환"""
    df = build_risk_ranking(ASOF_FOR_TOOLS, top_n=5)
    return df.to_string(index=False)



In [17]:
from crewai import Agent, Task, Crew, Process

def run_with_crewai(asof):
    global ASOF_FOR_TOOLS
    ASOF_FOR_TOOLS = asof

    # ---- Agents ----
    field_agent = Agent(
        role="현장 상태 수집·대행 Agent",
        goal="현장(라인/작업자/단계/진척)을 요약해 생산관리자에게 즉시 전달한다.",
        backstory="현장 이동/구두 확인을 대신해 로그와 스냅샷을 읽고, 상태를 한눈에 정리하는 슈퍼바이저.",
        allow_delegation=False,
        verbose=True,
        tools=[field_status_tool],
    )

    stage_alert_agent = Agent(
        role="공정 단계 알림 Agent",
        goal="계획 대비 공정 시작 지연/누락 가능성을 탐지하고 누구에게 어떤 알림을 보낼지 제안한다.",
        backstory="생산율 목표를 맞추기 위해, '지금 시작해야 할 단계'를 놓치지 않게 만든다.",
        allow_delegation=False,
        verbose=True,
        tools=[delayed_start_tool],
    )

    delay_pred_agent = Agent(
        role="지연·정체 예측 Agent",
        goal="pause/정체를 감지하고 마감(due_time) 위험과 필요한 조치를 제시한다.",
        backstory="라인 정체가 전체 생산율을 깎기 전에, 원인과 조치를 빠르게 제안한다.",
        allow_delegation=False,
        verbose=True,
        tools=[pause_tool, risk_rank_tool],
    )

    priority_agent = Agent(
        role="우선순위 재정렬 Agent",
        goal="현재 시점에서 관리자가 반드시 봐야 할 공정을 우선순위로 재정렬한다.",
        backstory="바쁜 순간에도 중요한 것부터 보게 만드는 '집중력 보조' 에이전트.",
        allow_delegation=False,
        verbose=True,
        tools=[risk_rank_tool],
    )

    # ---- Tasks ----
    t1 = Task(
        description="현장 요약을 10줄 이내로 정리하고, 현재 가장 위험한 공정 1개를 콕 집어라.",
        expected_output="현장 요약 + 최우선 주의 공정 1개(근거 포함)",
        agent=field_agent,
    )

    t2 = Task(
        description="계획 대비 지연 시작 또는 아직 시작하지 않은 공정을 찾고, 알림 메시지 초안을 작성하라.",
        expected_output="지연 공정 리스트 + 알림 메시지 초안(수신자: 라인단장/생산관리자)",
        agent=stage_alert_agent,
    )

    t3 = Task(
        description="pause/정체 위험을 분석하고, due_time 관점에서 즉시 조치해야 할 행동 3가지를 제안하라.",
        expected_output="정체 원인/영향 + 즉시 조치 3가지",
        agent=delay_pred_agent,
    )

    t4 = Task(
        description="현재 시점에서 생산관리자가 이동/확인해야 할 TOP3 우선순위를 표로 정리하라.",
        expected_output="TOP3 우선순위 표(PO, 공정, 이유, 다음 액션)",
        agent=priority_agent,
    )

    crew = Crew(
        agents=[field_agent, stage_alert_agent, delay_pred_agent, priority_agent],
        tasks=[t1, t2, t3, t4],
        process=Process.sequential,
        verbose=True
    )

    return crew.kickoff()

res = run_with_crewai(ASOF)
res


2026-01-15 13:25:53,232 - 134323151474816 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


 [DEBUG]: == Working Agent: 현장 상태 수집·대행 Agent
 [INFO]: == Starting Task: 현장 요약을 10줄 이내로 정리하고, 현재 가장 위험한 공정 1개를 콕 집어라.


> Entering new CrewAgentExecutor chain...
To complete the task, I need to use the available tool to gather the current status of the field, including the progress and any potential risks. This will help me identify the most critical process that requires attention.

Action: field_status
Action Input: {}
 

[현장 요약] 기준시각: 2026-01-15 14:00
- PO20260115-003(크림 50ml): 77.8% / 현재: 충진(Filling)(pause) / 라인:L2-충진 / 작업자:OP08 / 메모: 원료 용기 부족(자재 지연)
- PO20260115-001(클렌저 100ml): 100.0% / 현재: 라벨링(start) / 라인:L3-포장 / 작업자:OP04
- PO20260115-002(토너 200ml): 100.0% / 현재: 라벨링(start) / 라인:L3-포장 / 작업자:OP04
- PO20260115-004(세럼 30ml): 100.0% / 현재: 캡핑/실링(start) / 라인:L2-충진 / 작업자:OP03
- PO20260115-005(선크림 50ml): 100.0% / 현재: 캡핑/실링(start) / 라인:L2-충진 / 작업자:OP08

Thought: I now have the necessary information to summarize the field status and identify the most critical process. The process with the m

2026-01-15 13:26:21,055 - 134322250036928 - __init__.py-__init__:192 - ERROR: Exception while exporting Span.
Traceback (most recent call last):
  File "/home/yujin/skala/venvllm/lib/python3.11/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/home/yujin/skala/venvllm/lib/python3.11/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/http/client.py", line 1395, in getresponse
    response.begin()
  File "/usr/lib/python3.11/http/client.py", line 325, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/http/client.py", line 286, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socke

Thought: Based on the observations, the process for PO20260115-003 (SKU-CRM-50) is currently paused due to a material shortage, specifically the lack of raw material containers, which is a significant risk with a high risk score of 107.2. This poses an immediate threat to meeting the due time. Here are the causes and the necessary immediate actions:

Final Answer: 
정체 원인/영향: 현재 PO20260115-003 (SKU-CRM-50) 공정이 원료 용기 부족으로 인해 L2-충진 단계에서 정체되고 있습니다. 이 문제는 생산 일정에 직접적인 영향을 미치며, 특히 마감 시간(2026-01-15 14:00:00)까지 완료되지 않을 위험이 큽니다.

즉시 조치 3가지:
1. 자재팀에 즉시 연락하여 원료 용기의 긴급 공급을 요청합니다. 공급업체와 협의하여 가장 빠른 시간 내에 용기를 확보할 수 있도록 조치를 취해야 합니다.
2. 생산관리팀과 협력하여 다른 생산 라인의 자재를 재배치할 수 있는지 검토합니다. 만약 다른 라인의 진행에 큰 영향을 미치지 않는다면, 일시적으로 용기를 이동하여 사용합니다.
3. 정체로 인한 생산 차질을 최소화하기 위해 현재 작업 중인 다른 공정의 우선순위를 조정하여, 가능한 한 전체 생산 일정에 미치는 영향을 줄입니다.

> Finished chain.
 [DEBUG]: == [지연·정체 예측 Agent] Task output: 정체 원인/영향: 현재 PO20260115-003 (SKU-CRM-50) 공정이 원료 용기 부족으로 인해 L2-충진 단계에서 정체되고 있습니다. 이 문제는 생산 일정에 직접적인 영향을 미치며, 특히 마감 시간(2026-01-15 14:

'| PO            | 공정     | 이유                           | 다음 액션                                                                 |\n|---------------|----------|------------------------------|-------------------------------------------------------------------------|\n| PO20260115-003 | L2-충진 | 원료 용기 부족(자재 지연)     | 1. 자재팀에 긴급 공급 요청. 2. 다른 라인 자재 재배치 검토. 3. 우선순위 조정. |\n| PO20260115-004 | L2-충진 | 생산 진척도 100% 완료         | 정상 진행 중. 추가 조치 필요 없음.                                       |\n| PO20260115-001 | L3-포장 | 생산 진척도 100% 완료         | 정상 진행 중. 추가 조치 필요 없음.                                       |\n\nThis table lists the top three priorities that the production manager needs to focus on, with PO20260115-003 being the highest priority due to the risk of delay caused by a shortage of raw material containers.'

| PO            | 공정   | 이유                    | 다음 액션                           |
|---------------|--------|-------------------------|------------------------------------|
| PO20260115-003 | 충진   | 원료 용기 부족으로 중지 상태, 높은 위험 점수(107.2) | 1. 원료 용기 긴급 조달<br>2. 대체 자재 고려<br>3. 생산 일정 재조정 |
| PO20260115-004 | 충진   | 높은 위험 점수(45.0)     | 상황 모니터링 및 추가 조치 필요 여부 확인 |
| PO20260115-001 | 포장   | 위험 점수(35.0)          | 일정대로 진행 중, 지속적인 모니터링 필요 |

| PO            | 공정     | 이유                           | 다음 액션                                                                 |
|---------------|----------|------------------------------|-------------------------------------------------------------------------|
| PO20260115-003 | L2-충진 | 원료 용기 부족(자재 지연)     | 1. 자재팀에 긴급 공급 요청. 2. 다른 라인 자재 재배치 검토. 3. 우선순위 조정. |
| PO20260115-004 | L2-충진 | 생산 진척도 100% 완료         | 정상 진행 중. 추가 조치 필요 없음.                                       |
| PO20260115-001 | L3-포장 | 생산 진척도 100% 완료         | 정상 진행 중. 추가 조치 필요 없음.                                       |

This table lists the top three priorities that the production manager needs to focus on, with PO20260115-003 being the highest priority due to the risk of delay caused by a shortage of raw material containers.